# BTC YFinance Analysis

This notebook loads the `btc_yfinance_2015_2026.csv` export, runs the project's processing pipeline, and shows basic model evaluation. Cells include comments and are runnable in order.

In [ ]:
# Imports and path setup
import sys
from pathlib import Path
import pandas as pd

# Ensure repo root is on path so local modules import correctly
sys.path.insert(0, str(Path('.').resolve()))

# Display pandas version for reproducibility
print('pandas', pd.__version__)


In [ ]:
# Load the yfinance CSV and show top rows (the file should be in project root)
csv_path = 'btc_yfinance_2015_2026.csv'
raw = pd.read_csv(csv_path)
print('Raw columns:', raw.columns.tolist())
display(raw.head())

# Quick column presence check
for req in ['Date','Datetime','Open','High','Low','Close','Volume']:
    if req in raw.columns:
        print(f'Found: {req}')
    else:
        print(f'Missing (will attempt to parse alternatives): {req}')

In [ ]:
# Use the project's data processing functions to prepare the dataset
from btc_data_processing import load_clean_coinmarketcap_csv, add_time_features, add_technical_features, create_target, build_feature_matrix

# The loader is robust to some column name differences; pass the yfinance CSV path
df = load_clean_coinmarketcap_csv(csv_path)
print('Rows after cleaning:', len(df))
display(df.head())

In [ ]:
# Add features and create target, then build feature matrix used by models
df = add_time_features(df)  # adds day_of_week, month, is_weekend, etc.
df = add_technical_features(df)  # moving averages, RSI, returns
df = create_target(df, horizon=1)  # creates 'target_1d' column
X, y = build_feature_matrix(df, target_column='target_1d')
print('Rows available for modeling (after dropping NaNs):', len(X))
display(X.head())

In [ ]:
# Run the modeling pipeline to get baseline, linear, and RF results
from btc_modeling import run_pipeline, print_results
results = run_pipeline(csv_path)
print_results(results)